# 🏭 ENM412 – MAN Türkiye A.Ş. Stok Yönetimi
**Büşra ÇİL · İrem ÇELİK · Sevde SÖZDEN**

**Modeller:** RF · XGBoost · LightGBM · CatBoost + Grid Search + Optuna + SimPy

---
## Adımlar
1. Kütüphaneleri kur
2. Dosyaları yükle
3. Pipeline çalıştır (model eğitimi ~20-60 dk)
4. `enm412_cache.pkl` dosyasını indir
5. Cache'i GitHub'a yükle → Streamlit Cloud'da dashboard aç

## 1. Kütüphane Kurulumu

In [ ]:
!pip install optuna streamlit plotly xgboost lightgbm catboost openpyxl scikit-learn scipy simpy -q
print('✅ Kurulum tamamlandı')

## 2. Dosya Yükleme

Şu dosyaları seç:
- `MAN_ML_Dataset_v3.xlsx`
- `m1_veri.py`
- `m2_modeller.py`
- `m3_optimizasyon.py`
- `m4_pipeline.py`
- `dashboard.py`

In [ ]:
from google.colab import files
uploaded = files.upload()
print('\nYüklenen dosyalar:')
for f in uploaded:
    print(f'  ✅ {f}')

In [ ]:
import os
gerekli = ['MAN_ML_Dataset_v3.xlsx', 'm1_veri.py', 'm2_modeller.py',
           'm3_optimizasyon.py', 'm4_pipeline.py', 'dashboard.py']
print('Dosya kontrolü:')
for f in gerekli:
    print(f'  {"✅" if os.path.exists(f) else "❌ EKSİK"} {f}')

## 3. Pipeline – Model Eğitimi

⏱️ **Tahmini süre:** 20-60 dakika (segment sayısına ve n_trials'a göre)

Hızlı test için `--n_trials 10` kullanabilirsin.

In [ ]:
# Tam eğitim (önerilen - sunum için)
!python m4_pipeline.py --dosya MAN_ML_Dataset_v3.xlsx --n_trials 30 --cache enm412_cache.pkl

In [ ]:
# Hızlı test (5-10 dk)
# !python m4_pipeline.py --dosya MAN_ML_Dataset_v3.xlsx --n_trials 10 --cache enm412_cache.pkl

In [ ]:
import os
if os.path.exists('enm412_cache.pkl'):
    boyut = os.path.getsize('enm412_cache.pkl') / 1024 / 1024
    print(f'✅ Cache oluştu: {boyut:.1f} MB')
    print('Sıradaki adım: Cache dosyasını indir ve GitHub\'a yükle')
else:
    print('❌ Cache bulunamadı, pipeline hatalı tamamlandı')

## 4. Cache Dosyasını İndir

Bu dosyayı indirip GitHub reposuna yükle.
Streamlit Cloud dashboard'ı bu cache'i okuyarak çalışacak.

In [ ]:
from google.colab import files
files.download('enm412_cache.pkl')
print('✅ Cache indirildi!')
print('Şimdi GitHub reponuza yükleyin.')

## 5. Tekil Ürün Analizi (Opsiyonel)

Dashboard olmadan da doğrudan analiz yapabilirsin.

In [ ]:
import pickle, sys
sys.path.insert(0, '.')

with open('enm412_cache.pkl', 'rb') as f:
    sistem = pickle.load(f)

veri      = sistem['veri']
seg_mod   = sistem['seg_modelleri']
batch_df  = sistem['batch_df']

print(f'✅ {len(veri["parcalar"]):,} parça yüklendi')
print(f'Segmentler: {list(seg_mod.keys())}')
print(f'İlk 5 parça: {veri["parcalar"][:5]}')

In [ ]:
from m2_modeller import parca_tahmin
from m3_optimizasyon import parca_optimize, aksiyon_uyarisi

PID = 'Part-1'  # İstediğin parça ID'sini gir

# Tahmin
t = parca_tahmin(PID, veri['ml_df'], seg_mod, n_ay=6)

# Optimizasyon (Grid Search + Optuna + SimPy)
o = parca_optimize(PID, veri['opt_df'], veri['abc_df'],
                   t['tahminler'], grid_adim=12, n_trials=30, n_rep=15)

# Aksiyon uyarısı
uy = aksiyon_uyarisi(o, t['tahminler'])

print(f'=== {PID} ===')
print(f'Şampiyon Model : {t["sampiyon"]}')
samp_met = t['ml_metrikler'].get(t['sampiyon'], {})
print(f'MAE={samp_met.get("MAE",0):,.0f} | RMSE={samp_met.get("RMSE",0):,.0f} | MAPE={samp_met.get("MAPE",0):.1f}%')
print(f'\nOptimal: Q*={o["optimal_Q"]:,} | r*={o["optimal_r"]:,} | SS*={o["optimal_SS"]:,}')
print(f'SimPy Hizmet Düzeyi: %{o["sim_hizmet"]*100:.1f}')
print(f'\nTasarruf: {o["tasarruf_tl"]:,.0f} TL/ay ({o["tasarruf_oran"]:.1f}%)')
print(f'\n{uy["mesaj"]}')

In [ ]:
# Model karşılaştırma tablosu
import pandas as pd
ml_met  = t.get('ml_metrikler', {})
gel_met = t.get('gel_metrikler', {})
rows = []
for m_adi, m_val in {**ml_met, **gel_met}.items():
    rows.append({'Model': m_adi,
                 'MAE':  round(m_val.get('MAE', 0), 1),
                 'RMSE': round(m_val.get('RMSE', 0), 1),
                 'MAPE': f"{m_val.get('MAPE', 0):.1f}%"})
print(pd.DataFrame(rows).to_string(index=False))

---
**ENM412 – Endüstri Mühendisliğinde Tasarım II**
Büşra ÇİL · İrem ÇELİK · Sevde SÖZDEN | MAN Türkiye A.Ş. | 2024-2025